# SkyView Aerial Landscape — Ridge / Lasso Ontology Regression
**STA6704 Group Project**

Predicts the four class-derived ontology targets from tabular image features using **Ridge** and **Lasso**.

**Splits:** train/val/test membership comes from the EDA stratified **80/10/10** labels in `outputs/manifest.csv` (copied into the feature CSVs). This notebook asserts those labels still match; do not re-split here.

**Alpha selection:** each model picks `alpha` on **val** (lowest MAE, tie-break higher R²), then reports final metrics on **test**.

---
## 1. Imports

In [1]:
import os
import pathlib
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

PROJECT_ROOT = pathlib.Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "data_splits.py").is_file():
    raise FileNotFoundError(
        "Run this notebook from the repo root (or notebooks/) so src/data_splits.py is importable."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_splits import load_manifest

print("Imports OK.")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")

Imports OK.
PROJECT_ROOT : S:\Personal Projects\STA6704_Group_Project


---
## 2. Load artifacts & assert shared splits

Uses committed EDA outputs and checks that feature/target `split` labels match `outputs/manifest.csv`.

In [2]:
FEATURES_PATH = PROJECT_ROOT / "outputs" / "features" / "image_features.csv"
TARGETS_PATH = PROJECT_ROOT / "outputs" / "features" / "regression_targets.csv"
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "outputs" / "features" / "feature_columns.txt"
SCALER_PATH = PROJECT_ROOT / "outputs" / "features" / "scaler.joblib"
RESULTS_DIR = PROJECT_ROOT / "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

df_features = pd.read_csv(FEATURES_PATH)
df_targets = pd.read_csv(TARGETS_PATH)
manifest = load_manifest(PROJECT_ROOT)

with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
    feature_cols = [line.strip() for line in f if line.strip()]

missing_feats = [c for c in feature_cols if c not in df_features.columns]
if missing_feats:
    raise ValueError(f"feature_columns.txt entries missing from image_features.csv: {missing_feats}")

meta_cols = {"relative_path", "class_name", "label_idx", "split"}
target_cols = [c for c in df_targets.columns if c not in meta_cols]
expected_targets = [
    "is_human_made",
    "is_water_related",
    "is_vegetation_related",
    "urban_density_proxy",
]
if target_cols != expected_targets:
    raise ValueError(f"Unexpected target columns: {target_cols}; expected {expected_targets}")


def assert_split_alignment(df: pd.DataFrame, name: str) -> None:
    """Require df relative_path/split to match outputs/manifest.csv."""
    table_splits = df.set_index("relative_path")["split"]
    man_splits = manifest.set_index("relative_path")["split"]
    missing = man_splits.index.difference(table_splits.index)
    extra = table_splits.index.difference(man_splits.index)
    if len(missing) or len(extra):
        raise AssertionError(
            f"{name} relative_path mismatch vs manifest: "
            f"missing={len(missing)}, extra={len(extra)}"
        )
    aligned = table_splits.loc[man_splits.index]
    mismatched = aligned[aligned != man_splits]
    if len(mismatched):
        raise AssertionError(
            f"{name}: {len(mismatched)} rows have split labels that disagree with manifest.csv"
        )


assert_split_alignment(df_features, "image_features.csv")
assert_split_alignment(df_targets, "regression_targets.csv")

if not (
    df_features["relative_path"].equals(df_targets["relative_path"])
    and df_features["split"].equals(df_targets["split"])
):
    raise AssertionError("image_features.csv and regression_targets.csv are not row-aligned")

print("Split alignment OK: features and targets match manifest.csv on relative_path.")
print(f"Targets       : {target_cols}")
print(f"# predictors  : {len(feature_cols)}")

Split alignment OK: features and targets match manifest.csv on relative_path.
Targets       : ['is_human_made', 'is_water_related', 'is_vegetation_related', 'urban_density_proxy']
# predictors  : 43


---
## 3. Build train / val / test matrices

Scale with the committed train-fit `scaler.joblib` from EDA (do not refit).

In [3]:
train_mask = df_features["split"] == "train"
val_mask = df_features["split"] == "val"
test_mask = df_features["split"] == "test"

X_train_raw = df_features.loc[train_mask, feature_cols].to_numpy()
X_val_raw = df_features.loc[val_mask, feature_cols].to_numpy()
X_test_raw = df_features.loc[test_mask, feature_cols].to_numpy()

Y_train = df_targets.loc[train_mask, target_cols].to_numpy()
Y_val = df_targets.loc[val_mask, target_cols].to_numpy()
Y_test = df_targets.loc[test_mask, target_cols].to_numpy()

scaler = joblib.load(SCALER_PATH)
X_train = scaler.transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

print(
    f"Loaded train={X_train.shape[0]}, val={X_val.shape[0]}, test={X_test.shape[0]} "
    f"(fixed 80/10/10 from manifest)."
)
print(f"Feature matrix shape: {X_train.shape[1]} columns after scaling.")

Loaded train=9597, val=1200, test=1200 (fixed 80/10/10 from manifest).
Feature matrix shape: 43 columns after scaling.


---
## 4. Hyperparameters

Alpha grids are searched on **val** only. Final scores use **test**.

In [4]:
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]
LASSO_ALPHAS = [0.0001, 0.001, 0.01, 0.1]
RANDOM_SEED = 42

print(f"Ridge alphas : {RIDGE_ALPHAS}")
print(f"Lasso alphas : {LASSO_ALPHAS}")
print(f"random_state : {RANDOM_SEED}")

Ridge alphas : [0.1, 1.0, 10.0, 100.0]
Lasso alphas : [0.0001, 0.001, 0.01, 0.1]
random_state : 42


---
## 5. Train, tune on val, evaluate on test

In [5]:
def select_alpha(model_factory, alphas, X_tr, y_tr, X_va, y_va):
    """Pick alpha by lowest val MAE; tie-break higher val R²."""
    best = None
    for alpha in alphas:
        model = model_factory(alpha)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        mae = mean_absolute_error(y_va, preds)
        r2 = r2_score(y_va, preds)
        candidate = (mae, -r2, alpha, model)
        if best is None or candidate[:3] < best[:3]:
            best = candidate
    mae, neg_r2, alpha, _ = best
    return alpha, mae, -neg_r2


rows = []

for idx, target_name in enumerate(target_cols):
    print("=" * 41)
    print(f" TARGET: {target_name}")
    print("=" * 41)

    y_train = Y_train[:, idx]
    y_val = Y_val[:, idx]
    y_test = Y_test[:, idx]

    # --- Ridge ---
    ridge_alpha, ridge_val_mae, ridge_val_r2 = select_alpha(
        lambda a: Ridge(alpha=a),
        RIDGE_ALPHAS,
        X_train,
        y_train,
        X_val,
        y_val,
    )
    ridge = Ridge(alpha=ridge_alpha)
    ridge.fit(X_train, y_train)
    ridge_preds = ridge.predict(X_test)
    ridge_test_mae = mean_absolute_error(y_test, ridge_preds)
    ridge_test_r2 = r2_score(y_test, ridge_preds)

    print("  [Ridge]")
    print(f"    Chosen alpha (val): {ridge_alpha}")
    print(f"    Val  MAE / R²     : {ridge_val_mae:.4f} / {ridge_val_r2:.4f}")
    print(f"    Test MAE / R²     : {ridge_test_mae:.4f} / {ridge_test_r2:.4f}")

    # --- Lasso ---
    lasso_alpha, lasso_val_mae, lasso_val_r2 = select_alpha(
        lambda a: Lasso(alpha=a, random_state=RANDOM_SEED, max_iter=10000),
        LASSO_ALPHAS,
        X_train,
        y_train,
        X_val,
        y_val,
    )
    lasso = Lasso(alpha=lasso_alpha, random_state=RANDOM_SEED, max_iter=10000)
    lasso.fit(X_train, y_train)
    lasso_preds = lasso.predict(X_test)
    lasso_test_mae = mean_absolute_error(y_test, lasso_preds)
    lasso_test_r2 = r2_score(y_test, lasso_preds)

    zeroed = int(np.sum(lasso.coef_ == 0))
    kept = len(feature_cols) - zeroed

    print("\n  [Lasso]")
    print(f"    Chosen alpha (val): {lasso_alpha}")
    print(f"    Val  MAE / R²     : {lasso_val_mae:.4f} / {lasso_val_r2:.4f}")
    print(f"    Test MAE / R²     : {lasso_test_mae:.4f} / {lasso_test_r2:.4f}")
    print(f"    Feature selection : dropped {zeroed}, kept {kept} / {len(feature_cols)}")
    print()

    rows.append(
        {
            "target": target_name,
            "model": "Ridge",
            "alpha": ridge_alpha,
            "val_mae": ridge_val_mae,
            "val_r2": ridge_val_r2,
            "test_mae": ridge_test_mae,
            "test_r2": ridge_test_r2,
            "lasso_zeroed": np.nan,
            "lasso_kept": np.nan,
        }
    )
    rows.append(
        {
            "target": target_name,
            "model": "Lasso",
            "alpha": lasso_alpha,
            "val_mae": lasso_val_mae,
            "val_r2": lasso_val_r2,
            "test_mae": lasso_test_mae,
            "test_r2": lasso_test_r2,
            "lasso_zeroed": zeroed,
            "lasso_kept": kept,
        }
    )

metrics_df = pd.DataFrame(rows)
metrics_path = RESULTS_DIR / "ridge_lasso_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print(f"Saved metrics -> {metrics_path}")

 TARGET: is_human_made
  [Ridge]
    Chosen alpha (val): 0.1
    Val  MAE / R²     : 0.2526 / 0.5969
    Test MAE / R²     : 0.2617 / 0.5760

  [Lasso]
    Chosen alpha (val): 0.0001
    Val  MAE / R²     : 0.2535 / 0.5964
    Test MAE / R²     : 0.2630 / 0.5741
    Feature selection : dropped 5, kept 38 / 43

 TARGET: is_water_related
  [Ridge]
    Chosen alpha (val): 0.1
    Val  MAE / R²     : 0.2433 / 0.4677
    Test MAE / R²     : 0.2447 / 0.4539

  [Lasso]
    Chosen alpha (val): 0.0001
    Val  MAE / R²     : 0.2449 / 0.4621
    Test MAE / R²     : 0.2460 / 0.4465
    Feature selection : dropped 4, kept 39 / 43

 TARGET: is_vegetation_related
  [Ridge]
    Chosen alpha (val): 100.0
    Val  MAE / R²     : 0.2099 / 0.4578
    Test MAE / R²     : 0.2074 / 0.4600

  [Lasso]
    Chosen alpha (val): 0.001
    Val  MAE / R²     : 0.2098 / 0.4543
    Test MAE / R²     : 0.2063 / 0.4608
    Feature selection : dropped 14, kept 29 / 43

 TARGET: urban_density_proxy
  [Ridge]
    Chosen a

---
## 6. Summary table

In [6]:
display_cols = [
    "target",
    "model",
    "alpha",
    "val_mae",
    "val_r2",
    "test_mae",
    "test_r2",
    "lasso_zeroed",
    "lasso_kept",
]
summary = metrics_df[display_cols].copy()
for col in ["val_mae", "val_r2", "test_mae", "test_r2"]:
    summary[col] = summary[col].map(lambda x: f"{x:.4f}")

summary

,target,model,alpha,val_mae,val_r2,test_mae,test_r2,lasso_zeroed,lasso_kept
0,is_human_made,Ridge,0.1000,0.2526,0.5969,0.2617,0.5760,NaN,NaN
1,is_human_made,Lasso,0.0001,0.2535,0.5964,0.2630,0.5741,5.0,38.0
2,is_water_related,Ridge,0.1000,0.2433,0.4677,0.2447,0.4539,NaN,NaN
3,is_water_related,Lasso,0.0001,0.2449,0.4621,0.2460,0.4465,4.0,39.0
4,is_vegetation_related,Ridge,100.0000,0.2099,0.4578,0.2074,0.4600,NaN,NaN
5,is_vegetation_related,Lasso,0.0010,0.2098,0.4543,0.2063,0.4608,14.0,29.0
6,urban_density_proxy,Ridge,1.0000,0.5439,0.5911,0.5669,0.5694,NaN,NaN
7,urban_density_proxy,Lasso,0.0001,0.5441,0.5909,0.5669,0.5695,4.0,39.0
